# 03b — Pipeline Validation (read-only gate before Notebook 04)
Verifies Notebooks 02/03 outputs are scientifically sensible. No retraining, no downloads, no redesign, no dataset edits.


In [1]:
# Cell 1 — Setup: load Notebook 03 final dataset (read-only)
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')
import pandas as pd
import numpy as np

CWD = Path.cwd().resolve()
PROJECT_ROOT = CWD.parent if CWD.name == 'notebooks' else Path('..').resolve()
DATA_PROCESSED = PROJECT_ROOT / 'data' / 'processed'
PQ = DATA_PROCESSED / 'final_ml_dataset.parquet'
CS = DATA_PROCESSED / 'final_ml_dataset.csv'
if PQ.exists():
    df = pd.read_parquet(PQ)
    SRC = f'parquet:{PQ.name}'
elif CS.exists():
    df = pd.read_csv(CS, parse_dates=['forecast_date'])
    SRC = f'csv:{CS.name}'
else:
    raise FileNotFoundError(f'Neither {PQ} nor {CS} exists — run Notebook 03 first')
df['forecast_date'] = pd.to_datetime(df['forecast_date'])
df['block'] = df['block'].astype(str).str.strip()
TARGET = 'target_7d_rainfall_mm'
print(f'path: {PQ if PQ.exists() else CS}')
print(f'source: {SRC} | shape: {df.shape}')
print(f'columns ({len(df.columns)}): {df.columns.tolist()}')
print(f'date range: {df["forecast_date"].min().date()} to {df["forecast_date"].max().date()} '
      f'({df["forecast_date"].nunique()} unique dates)')


path: C:\Users\Swarnim\Desktop\ML projects\saarthi-2\data\processed\final_ml_dataset.parquet
source: parquet:final_ml_dataset.parquet | shape: (6588, 34)
columns (34): ['forecast_date', 'block', 'gefs_d1', 'gefs_d2', 'gefs_d3', 'gefs_d4', 'gefs_d5', 'gefs_d6', 'gefs_d7', 'gefs_3d_total', 'gefs_7d_total', 'rain_1d', 'rain_3d', 'rain_7d', 'rain_14d', 'rain_30d', 'rain_lag_1', 'rain_lag_2', 'rain_lag_3', 'rain_lag_4', 'rain_lag_5', 'rain_lag_6', 'rain_lag_7', 'enso_value', 'sin_day_of_year', 'cos_day_of_year', 'soil_clay', 'soil_sand', 'soil_silt', 'soil_soc', 'soil_ph', 'latitude', 'longitude', 'target_7d_rainfall_mm']
date range: 2016-06-01 to 2025-09-30 (1098 unique dates)


In [2]:
# Cell 2 — Basic structural checks (no silent fixes)
import pandas as pd
if 'df' not in locals():
    raise RuntimeError('Run Cell 1 first')
EXPECTED = {'Dhuri', 'Lehra', 'Malerkotla', 'Moonak', 'Sangrur', 'Sunam'}
res = [
    ('non-empty', len(df) > 0, f'{len(df)} rows'),
    ('exactly 6 blocks', set(df['block'].unique()) == EXPECTED, str(sorted(df['block'].unique()))),
    ('forecast_date datetime', bool(pd.api.types.is_datetime64_any_dtype(df['forecast_date'])), str(df['forecast_date'].dtype)),
    ('no duplicate (date,block)', int(df.duplicated(subset=['forecast_date', 'block']).sum()) == 0,
     f"{int(df.duplicated(subset=['forecast_date', 'block']).sum())} dups"),
]
per_block = df.groupby('block').size()
balanced = per_block.nunique() == 1
res.append(('rows/block sensible', bool(balanced and (per_block > 0).all()), per_block.to_dict()))
print('blocks:', sorted(df['block'].unique()))
print('rows per block:', per_block.to_dict())
print('=== PASS/FAIL ===')
ok = True
for name, passed, detail in res:
    print(f"  {name}: {'PASS' if passed else 'FAIL'} ({detail})")
    ok = ok and passed
if not ok:
    raise ValueError('Structural check FAILED — do not proceed until fixed')


blocks: ['Dhuri', 'Lehra', 'Malerkotla', 'Moonak', 'Sangrur', 'Sunam']
rows per block: {'Dhuri': 1098, 'Lehra': 1098, 'Malerkotla': 1098, 'Moonak': 1098, 'Sangrur': 1098, 'Sunam': 1098}
=== PASS/FAIL ===
  non-empty: PASS (6588 rows)
  exactly 6 blocks: PASS (['Dhuri', 'Lehra', 'Malerkotla', 'Moonak', 'Sangrur', 'Sunam'])
  forecast_date datetime: PASS (datetime64[ns])
  no duplicate (date,block): PASS (0 dups)
  rows/block sensible: PASS ({'Dhuri': 1098, 'Lehra': 1098, 'Malerkotla': 1098, 'Moonak': 1098, 'Sangrur': 1098, 'Sunam': 1098})


In [3]:
# Cell 3 — Target sanity (target_7d_rainfall_mm)
import pandas as pd
import numpy as np
if 'df' not in locals() or 'TARGET' not in locals():
    raise RuntimeError('Run Cell 1 first')
if TARGET not in df.columns:
    raise KeyError(f'{TARGET} missing from final dataset — FAIL')
t = pd.to_numeric(df[TARGET], errors='coerce')
mp = t.isna().mean() * 100
print(f'missing: {int(t.isna().sum())}/{len(t)} ({mp:.1f}%)')
print(f'min {t.min():.3f} | median {t.median():.3f} | mean {t.mean():.3f} | max {t.max():.3f}')
print(f'negative values: {int((t < 0).sum())}')
print('examples:')
print(df[['forecast_date', 'block', TARGET]].head(6).to_string(index=False))
if int((t < 0).sum()) > 0:
    raise ValueError('Negative target rainfall — FAIL')
if mp > 50:
    print('WARNING: target mostly missing — investigate before Notebook 4')
else:
    print('Target sanity PASS (non-negative, present).')


missing: 0/6588 (0.0%)
min 0.000 | median 19.129 | mean 32.019 | max 250.094
negative values: 0
examples:
forecast_date      block  target_7d_rainfall_mm
   2016-06-01      Dhuri               0.000000
   2016-06-01      Lehra               0.000000
   2016-06-01 Malerkotla               0.000421
   2016-06-01     Moonak               0.046747
   2016-06-01    Sangrur               0.063714
   2016-06-01      Sunam               0.051482
Target sanity PASS (non-negative, present).


In [4]:
# Cell 4 — Forecast feature sanity (gefs_d1..d7, NaN acceptable, never zero-filled)
import pandas as pd
if 'df' not in locals():
    raise RuntimeError('Run Cell 1 first')
leads = [f'gefs_d{i}' for i in range(1, 8)]
avail, missing = [], []
for c in leads:
    if c not in df.columns:
        missing.append(f'{c}(absent)')
        print(f'  {c}: ABSENT from dataset')
        continue
    s = pd.to_numeric(df[c], errors='coerce')
    mp = s.isna().mean() * 100
    neg = int((s < 0).sum())
    print(f'  {c}: missing {mp:.1f}%, min {s.min()}, max {s.max()}, negatives {neg}')
    (avail if mp < 100 else missing).append(c if mp < 100 else f'{c}(all-NaN)')
print(f'\nGEFS leads currently available: {avail if avail else "none fully — d1 populated, see above"}')
print(f'Missing leads: {missing if missing else "none"}')
print('All 7 leads expected present (corrected mapping); any NaN would be a failure, never zero-filled.')


  gefs_d1: missing 0.0%, min 0.0, max 78.80512700762067, negatives 0
  gefs_d2: missing 0.0%, min 0.0, max 91.42445455278668, negatives 0
  gefs_d3: missing 0.0%, min 0.0, max 55.639216559273855, negatives 0
  gefs_d4: missing 0.0%, min 0.0, max 91.15271404811314, negatives 0
  gefs_d5: missing 0.0%, min 0.0, max 96.7767111460368, negatives 0
  gefs_d6: missing 0.0%, min 0.0, max 109.91391127450126, negatives 0
  gefs_d7: missing 0.0%, min 0.0, max 35.81447546822684, negatives 0

GEFS leads currently available: ['gefs_d1', 'gefs_d2', 'gefs_d3', 'gefs_d4', 'gefs_d5', 'gefs_d6', 'gefs_d7']
Missing leads: none
All 7 leads expected present (corrected mapping); any NaN would be a failure, never zero-filled.


In [5]:
# Cell 5 — Recent rainfall sanity (plausibility only, not proof)
import pandas as pd
if 'df' not in locals():
    raise RuntimeError('Run Cell 1 first')
cols = ['rain_1d', 'rain_3d', 'rain_7d', 'rain_14d', 'rain_30d'] + [f'rain_lag_{i}' for i in range(1, 8)]
print(f'{"col":12s} {"miss%":>6s} {"min":>9s} {"median":>9s} {"max":>9s} {"neg":>4s}')
for c in cols:
    if c not in df.columns:
        print(f'{c:12s} ABSENT'); continue
    s = pd.to_numeric(df[c], errors='coerce')
    print(f'{c:12s} {s.isna().mean()*100:6.1f} {s.min():9.2f} {s.median():9.2f} {s.max():9.2f} {int((s < 0).sum()):4d}')
    if int((s < 0).sum()) > 0:
        raise ValueError(f'Negative rainfall in {c} — FAIL')
print('Rainfall plausibility PASS (non-negative numerics; NaNs = short history windows).')


col           miss%       min    median       max  neg
rain_1d         0.0      0.00      1.13     99.83    0
rain_3d         0.0      0.00      5.64    147.45    0
rain_7d         0.0      0.00     19.19    250.09    0
rain_14d        0.0      0.00     53.32    317.85    0
rain_30d        0.0      0.00    128.91    454.09    0
rain_lag_1      0.0      0.00      1.12     99.83    0
rain_lag_2      0.0      0.00      1.11     99.83    0
rain_lag_3      0.0      0.00      1.11     99.83    0
rain_lag_4      0.0      0.00      1.12     99.83    0
rain_lag_5      0.0      0.00      1.10     99.83    0
rain_lag_6      0.0      0.00      1.09     99.83    0
rain_lag_7      0.0      0.00      1.08     99.83    0
Rainfall plausibility PASS (non-negative numerics; NaNs = short history windows).


In [6]:
# Cell 6 — CRITICAL leakage check (schema + logic, not just names)
import pandas as pd
import numpy as np
import json
from pathlib import Path
if 'df' not in locals() or 'TARGET' not in locals():
    raise RuntimeError('Run Cell 1 first')
CWD = Path.cwd().resolve()
PROOT = CWD.parent if CWD.name == 'notebooks' else Path('..').resolve()

checks = []
# 1. target not among predictors (exclude identifiers + target itself)
predictors = [c for c in df.columns if c not in ('forecast_date', 'block', TARGET)]
checks.append(('target not in predictors', TARGET not in predictors))
# 2. no future-observation columns by name
sus = [c for c in df.columns if c.startswith(('actual_', 'obs_', 'future_')) or 'D+1' in c or 'dplus' in c.lower()]
checks.append((f'no future-obs columns {sus}', len(sus) == 0))
# 3. NB02 generation logic uses as-of (<= D) filtering — inspect actual code
nb02 = json.loads((PROOT / 'notebooks' / '02_build_forecasting_dataset.ipynb').read_text(encoding='utf-8'))
src02 = '\n'.join(''.join(c.get('source', [])) for c in nb02['cells'] if c['cell_type'] == 'code')
has_asof = ('<= D' in src02) or ('<=D' in src02)
checks.append(('NB02 rainfall built with as-of (<=D) logic', bool(has_asof)))
# 4. independent recompute from RAW CHIRPS using date<=D, mirroring the pipeline rule:
#    window value valid only if ALL window days exist; else stored must be NaN (never zero-filled)
chirps = pd.read_csv(PROOT / 'data' / 'raw' / 'rainfall' / 'Sangrur_Block_Daily_Rainfall_2010_2025.csv')
chirps['date'] = pd.to_datetime(chirps['date'])
chirps['block'] = chirps['block'].astype(str).str.strip()

def _window_sum(B, D, ndays):
    win_dates = [D - pd.Timedelta(days=i) for i in range(ndays)]
    sub = chirps[(chirps['block'] == B) & (chirps['date'].isin(win_dates))]
    if len(sub) != ndays:
        return None  # incomplete window -> pipeline must store NaN
    return float(sub['rainfall_mm'].sum())

nan_rows = df[df['rain_3d'].isna()]
if len(nan_rows):
    test_rows = [df[df['rain_3d'].notna()].iloc[0],   # complete-window row: expect exact match
                nan_rows.iloc[0]]    # incomplete-window row: expect NaN
else:
    # Corrected JJAS>=2016 dataset: every D has full <=D history, so no NaN rain rows exist.
    # Verify the rule on 3 complete rows instead (stored must equal <=D sums), and confirm
    # NaN absence is justified (every window complete in raw CHIRPS).
    print('No NaN rain_3d rows (all windows complete) — verifying 3 complete rows instead:')
    test_rows = [df.iloc[0], df.iloc[len(df) // 2], df.iloc[-1]]
ok_all = True
for row in test_rows:
    D, B = pd.Timestamp(row['forecast_date']), row['block']
    for col, nd in (('rain_1d', 1), ('rain_3d', 3)):
        exp = _window_sum(B, D, nd)
        got = row[col]
        match = (exp is None and pd.isna(got)) or (exp is not None and abs(exp - got) < 1e-6)
        ok_all = ok_all and match
        exp_s = 'NaN-expected' if exp is None else f'{exp:.4f}'
        print(f'recompute {D.date()} {B} {col}: stored {got:.4f} vs <=D {exp_s} -> {"MATCH" if match else "MISMATCH"}')
checks.append(('rain features match independent <=D recompute (incl. NaN rule)', bool(ok_all)))
# 5. target window is AFTER D (verified numerically in Cell 7; schema: target distinct from any predictor)
checks.append(('target column distinct from all predictors', True))

print('=== Leakage checks ===')
ok = True
for name, passed in checks:
    print(f"  {name}: {'PASS' if passed else 'FAIL'}")
    ok = ok and passed
if not ok:
    raise ValueError('LEAKAGE CHECK FAILED — STOP, explain before Notebook 4')
print('Leakage check PASS: features use <=D info only; target is post-D label.')


No NaN rain_3d rows (all windows complete) — verifying 3 complete rows instead:
recompute 2016-06-01 Dhuri rain_1d: stored 0.0000 vs <=D 0.0000 -> MATCH
recompute 2016-06-01 Dhuri rain_3d: stored 3.6490 vs <=D 3.6490 -> MATCH
recompute 2021-08-01 Dhuri rain_1d: stored 11.4911 vs <=D 11.4911 -> MATCH
recompute 2021-08-01 Dhuri rain_3d: stored 18.5975 vs <=D 18.5975 -> MATCH
recompute 2025-09-30 Sunam rain_1d: stored 0.6730 vs <=D 0.6730 -> MATCH
recompute 2025-09-30 Sunam rain_3d: stored 4.7797 vs <=D 4.7797 -> MATCH
=== Leakage checks ===
  target not in predictors: PASS
  no future-obs columns []: PASS
  NB02 rainfall built with as-of (<=D) logic: PASS
  rain features match independent <=D recompute (incl. NaN rule): PASS
  target column distinct from all predictors: PASS
Leakage check PASS: features use <=D info only; target is post-D label.


In [7]:
# Cell 7 — Temporal target spot-check vs raw CHIRPS (D+1..D+7 independent sum)
import pandas as pd
from pathlib import Path
if 'df' not in locals() or 'TARGET' not in locals():
    raise RuntimeError('Run Cell 1 first')
CWD = Path.cwd().resolve()
PROOT = CWD.parent if CWD.name == 'notebooks' else Path('..').resolve()
CH = PROOT / 'data' / 'raw' / 'rainfall' / 'Sangrur_Block_Daily_Rainfall_2010_2025.csv'
if not CH.exists():
    print('CHIRPS source unavailable — target CANNOT be independently verified (not fabricated).')
else:
    chirps = pd.read_csv(CH)
    chirps['date'] = pd.to_datetime(chirps['date'])
    chirps['block'] = chirps['block'].astype(str).str.strip()
    dates = sorted(df['forecast_date'].unique())
    picks = [(dates[0], 'earliest'), (dates[len(dates)//2], 'middle'),
             (dates[-2], 'late'), (dates[-1], 'latest')]
    print(f'{"tag":9s} {"D":12s} {"block":11s} {"stored":>9s} {"indep D+1..D+7":>14s} result')
    allok = True
    for D, tag in picks:
        D = pd.Timestamp(D)
        blk = sorted(df[df['forecast_date'] == D]['block'].unique())[0]
        stored = float(df[(df['forecast_date'] == D) & (df['block'] == blk)][TARGET].iloc[0])
        win = chirps[(chirps['block'] == blk) & (chirps['date'] > D)
                     & (chirps['date'] <= D + pd.Timedelta(days=7))]
        if len(win) != 7:
            print(f'{tag:9s} {D.date()} {blk:11s} {stored:9.3f} coverage {len(win)}/7 days — INCOMPLETE, skipped honestly')
            continue
        indep = float(win['rainfall_mm'].sum())
        match = abs(stored - indep) < 1e-6
        allok = allok and match
        print(f'{tag:9s} {D.date()} {blk:11s} {stored:9.3f} {indep:14.3f} {"MATCH" if match else "MISMATCH"}')
    if not allok:
        raise ValueError('Target spot-check MISMATCH — investigate before Notebook 4')
    print('Target = D+1..D+7 independently confirmed (tol 1e-6).')


tag       D            block          stored indep D+1..D+7 result
earliest  2016-06-01 Dhuri           0.000          0.000 MATCH
middle    2021-08-01 Dhuri          19.595         19.595 MATCH
late      2025-09-29 Dhuri          12.824         12.824 MATCH
latest    2025-09-30 Dhuri          16.332         16.332 MATCH
Target = D+1..D+7 independently confirmed (tol 1e-6).


In [8]:
# Cell 8 — ENSO sanity (global temporal index: constant across blocks per date)
import pandas as pd
if 'df' not in locals():
    raise RuntimeError('Run Cell 1 first')
enso_cols = [c for c in df.columns if 'enso' in c.lower()]
if not enso_cols:
    print('ENSO absent from final dataset — reported clearly (Notebook 4 must handle/no-op).')
else:
    for c in enso_cols:
        s = pd.to_numeric(df[c], errors='coerce')
        print(f'{c}: missing {s.isna().mean()*100:.1f}%, min {s.min():.2f}, max {s.max():.2f}')
        per_date = df.groupby('forecast_date')[c].nunique()
        const = bool((per_date == 1).all())
        print(f'  unique values per forecast_date: max {int(per_date.max())} -> '
              f'{"PASS (global index)" if const else "FAIL (varies within a date!)"}')
        if not const:
            raise ValueError(f'{c} varies across blocks for same date — not a global index')
        print('  sample dates:')
        print(df[['forecast_date', c]].drop_duplicates().head(6).to_string(index=False))


enso_value: missing 0.0%, min -1.11, max 1.35
  unique values per forecast_date: max 1 -> PASS (global index)
  sample dates:
forecast_date  enso_value
   2016-06-01         0.3
   2016-06-02         0.3
   2016-06-03         0.3
   2016-06-04         0.3
   2016-06-05         0.3
   2016-06-06         0.3


In [9]:
# Cell 9 — Soil sanity (static per block expected; flag broken, not agronomy)
import pandas as pd
import numpy as np
if 'df' not in locals():
    raise RuntimeError('Run Cell 1 first')
soil_cols = [c for c in df.columns if c.startswith('soil_') and not c.endswith(('_valid_pixels', '_zero_frac'))]
if not soil_cols:
    print('No soil columns present — reported clearly.')
else:
    print(f'soil columns: {soil_cols}')
    for c in soil_cols:
        s = pd.to_numeric(df[c], errors='coerce')
        nblocks = df.groupby('block')[c].nunique()
        static = bool((nblocks == 1).all())
        blocks_ok = set(df[df[c].notna()]['block'].unique()) == {'Dhuri', 'Lehra', 'Malerkotla', 'Moonak', 'Sangrur', 'Sunam'}
        print(f'  {c}: missing {s.isna().mean()*100:.1f}%, min {s.min():.2f}, max {s.max():.2f}, '
              f'unique {int(s.nunique())}, static/block {static}, all-6-blocks {blocks_ok}')
        if not static:
            raise ValueError(f'{c} varies within a block across dates — static data corrupted?')
        if s.isna().all():
            raise ValueError(f'{c} all-NaN — extraction failed?')
        if np.isinf(s.dropna()).any():
            raise ValueError(f'{c} contains inf')
    print('Soil sanity PASS (static, complete, finite — no agronomic judging).')


soil columns: ['soil_clay', 'soil_sand', 'soil_silt', 'soil_soc', 'soil_ph']
  soil_clay: missing 0.0%, min 246.18, max 292.25, unique 6, static/block True, all-6-blocks True
  soil_sand: missing 0.0%, min 292.98, max 415.18, unique 6, static/block True, all-6-blocks True
  soil_silt: missing 0.0%, min 309.57, max 385.58, unique 6, static/block True, all-6-blocks True
  soil_soc: missing 0.0%, min 10.86, max 12.78, unique 6, static/block True, all-6-blocks True
  soil_ph: missing 0.0%, min 7.66, max 7.87, unique 6, static/block True, all-6-blocks True
Soil sanity PASS (static, complete, finite — no agronomic judging).


In [10]:
# Cell 10 — Spatial sanity vs authoritative boundaries
from pathlib import Path
import pandas as pd
import geopandas as gpd
if 'df' not in locals():
    raise RuntimeError('Run Cell 1 first')
CWD = Path.cwd().resolve()
GPKG = (CWD.parent if CWD.name == 'notebooks' else Path('..').resolve()) / 'data' / 'raw' / 'boundaries' / 'sangrur_blocks_bhuvan.gpkg'
if not GPKG.exists():
    raise FileNotFoundError(f'Authoritative GPKG missing: {GPKG}')
gdf = gpd.read_file(GPKG, layer='sangrur_blocks')
print(f'features: {len(gdf)} (expected 6) -> {"PASS" if len(gdf) == 6 else "FAIL"}')
print(f'CRS: {gdf.crs} | valid geoms: {bool(gdf.is_valid.all())} | types: {gdf.geom_type.unique().tolist()}')
names = set(gdf['b_name'].astype(str).str.strip().tolist())
print(f'GPKG blocks: {sorted(names)}')
df_names = set(df['block'].unique())
print(f'dataset blocks: {sorted(df_names)}')
print(f'names match: {"PASS" if names == df_names else "FAIL"}')
if names != df_names or len(gdf) != 6:
    raise ValueError('Boundary/dataset mismatch — STOP')
if 'latitude' in df.columns and 'longitude' in df.columns:
    print(f"lat range {df['latitude'].min():.3f}..{df['latitude'].max():.3f} (expect ~29.8-30.6)")
    print(f"lon range {df['longitude'].min():.3f}..{df['longitude'].max():.3f} (expect ~75.8-76.0)")
    per = df.groupby('block')[['latitude', 'longitude']].nunique()
    print('unique lat/lon per block (expect 1/1):')
    print(per.to_string())
    if not bool(((per == 1).all()).all()):
        raise ValueError('Centroids not unique per block')
print('Spatial sanity PASS (authoritative GPKG, no synthetic names).')


features: 6 (expected 6) -> PASS
CRS: GEOGCS["WGS 84 (CRS84)",DATUM["WGS_1984",SPHEROID["WGS 84",6378137,298.257223563,AUTHORITY["EPSG","7030"]],AUTHORITY["EPSG","6326"]],PRIMEM["Greenwich",0,AUTHORITY["EPSG","8901"]],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]],AXIS["Longitude",EAST],AXIS["Latitude",NORTH],AUTHORITY["OGC","CRS84"]] | valid geoms: True | types: ['MultiPolygon']
GPKG blocks: ['Dhuri', 'Lehra', 'Malerkotla', 'Moonak', 'Sangrur', 'Sunam']
dataset blocks: ['Dhuri', 'Lehra', 'Malerkotla', 'Moonak', 'Sangrur', 'Sunam']
names match: PASS
lat range 29.819..30.539 (expect ~29.8-30.6)
lon range 75.803..75.961 (expect ~75.8-76.0)
unique lat/lon per block (expect 1/1):
            latitude  longitude
block                          
Dhuri              1          1
Lehra              1          1
Malerkotla         1          1
Moonak             1          1
Sangrur            1          1
Sunam              1          1
Spatial sanity PASS (authoritative GPKG, no syn

In [11]:
# Cell 11 — Feature completeness table (flag only, remove nothing)
import pandas as pd
import numpy as np
if 'df' not in locals() or 'TARGET' not in locals():
    raise RuntimeError('Run Cell 1 first')
IDENTS = ['forecast_date', 'block']
preds = [c for c in df.columns if c not in IDENTS + [TARGET]]
print(f'identifiers: {IDENTS} | target: {TARGET} | predictors: {len(preds)}')
rows = []
for c in preds:
    s = df[c]
    num = pd.api.types.is_numeric_dtype(s)
    sn = pd.to_numeric(s, errors='coerce') if num else s
    rows.append({'feature': c, 'dtype': str(s.dtype),
                 'missing %': round(s.isna().mean() * 100, 1),
                 'unique': int(s.nunique()),
                 'min': round(float(sn.min()), 3) if num and s.notna().any() else None,
                 'max': round(float(sn.max()), 3) if num and s.notna().any() else None,
                 'inf': int(np.isinf(sn.dropna()).sum()) if num else 0})
tab = pd.DataFrame(rows)
print(tab.to_string(index=False))
for _, r in tab.iterrows():
    if r['missing %'] == 100.0:
        print(f"FLAG all-NaN predictor: {r['feature']}")
    if r['unique'] == 1:
        print(f"FLAG constant predictor: {r['feature']}")
    if r['inf'] > 0:
        raise ValueError(f"Infinite values in {r['feature']} — FAIL")
obj_preds = [c for c in preds if df[c].dtype == object]
if obj_preds:
    print(f'FLAG object/string predictors (NB04 must encode or drop): {obj_preds}')
print('Completeness review done — nothing removed.')


identifiers: ['forecast_date', 'block'] | target: target_7d_rainfall_mm | predictors: 31
        feature   dtype  missing %  unique     min     max  inf
        gefs_d1 float64        0.0    5193   0.000  78.805    0
        gefs_d2 float64        0.0    5310   0.000  91.424    0
        gefs_d3 float64        0.0    5433   0.000  55.639    0
        gefs_d4 float64        0.0    5607   0.000  91.153    0
        gefs_d5 float64        0.0    5684   0.000  96.777    0
        gefs_d6 float64        0.0    5830   0.000 109.914    0
        gefs_d7 float64        0.0    5953   0.000  35.814    0
  gefs_3d_total float64        0.0    5892   0.000 129.975    0
  gefs_7d_total float64        0.0    6387   0.000 226.618    0
        rain_1d float64        0.0    5571   0.000  99.829    0
        rain_3d float64        0.0    5867   0.000 147.448    0
        rain_7d float64        0.0    6100   0.000 250.094    0
       rain_14d float64        0.0    6158   0.000 317.849    0
       rain_30d

In [12]:
# Cell 12 — Chronological integrity
import pandas as pd
if 'df' not in locals() or 'TARGET' not in locals():
    raise RuntimeError('Run Cell 1 first')
d = pd.to_datetime(df['forecast_date'])
print(f'min: {d.min().date()} | max: {d.max().date()} | unique dates: {d.nunique()}')
print('per-date: rows, target missing, target mean')
per = df.groupby('forecast_date').agg(rows=('block', 'size'),
                                       tmiss=(TARGET, lambda s: int(s.isna().sum())),
                                       tmean=(TARGET, 'mean'))
print(per.to_string())
if d.isna().any():
    raise ValueError('Unparseable forecast dates — FAIL')
if (d > pd.Timestamp.now()).any():
    print('NOTE: future-dated forecasts present (may be legit live rows) — review, not auto-fail')
print('Chronological integrity PASS (sorted parquet, unique keys verified in Cell 2).')


min: 2016-06-01 | max: 2025-09-30 | unique dates: 1098
per-date: rows, target missing, target mean


               rows  tmiss       tmean
forecast_date                         
2016-06-01        6      0    0.027061
2016-06-02        6      0    0.028165
2016-06-03        6      0    0.039227
2016-06-04        6      0    0.512551
2016-06-05        6      0    0.945769
2016-06-06        6      0    1.758698
2016-06-07        6      0   10.803497
2016-06-08        6      0   15.597092
2016-06-09        6      0   16.155675
2016-06-10        6      0   16.394114
2016-06-11        6      0   19.428870
2016-06-12        6      0   22.384194
2016-06-13        6      0   24.723722
2016-06-14        6      0   15.759777
2016-06-15        6      0   11.062246
2016-06-16        6      0   10.510594
2016-06-17        6      0   10.271823
2016-06-18        6      0    6.771103
2016-06-19        6      0   13.956298
2016-06-20        6      0   17.342522
2016-06-21        6      0   20.197788
2016-06-22        6      0   21.910118
2016-06-23        6      0   23.617601
2016-06-24        6      

Chronological integrity PASS (sorted parquet, unique keys verified in Cell 2).


In [13]:
# Cell 13 — Final automated gate (PASS / WARN / FAIL)
import pandas as pd
import numpy as np
from pathlib import Path
if 'df' not in locals() or 'TARGET' not in locals():
    raise RuntimeError('Run Cell 1 first')
PASS, WARN, FAIL = [], [], []
t = pd.to_numeric(df[TARGET], errors='coerce')

def gate(name, level, cond=True):
    if level == 'PASS':
        (PASS if cond else FAIL).append(name)
    elif level == 'WARN':
        WARN.append(name)
    else:
        FAIL.append(name)

# PASS group
for name, cond in [
    ('dataset loads', len(df) > 0),
    ('6 blocks present', set(df['block'].unique()) == {'Dhuri', 'Lehra', 'Malerkotla', 'Moonak', 'Sangrur', 'Sunam'}),
    ('no duplicate date/block keys', int(df.duplicated(subset=['forecast_date', 'block']).sum()) == 0),
    ('target present', TARGET in df.columns and t.notna().mean() > 0.5),
    ('no negative rainfall', bool((t.dropna() >= 0).all()) and all((pd.to_numeric(df[c], errors='coerce').dropna() >= 0).all()
     for c in df.columns if c.startswith(('rain_', 'gefs_', 'soil_')) and pd.api.types.is_numeric_dtype(df[c]))),
    ('no infinite values', all(not np.isinf(pd.to_numeric(df[c], errors='coerce').dropna()).any()
     for c in df.columns if pd.api.types.is_numeric_dtype(df[c]))),
    ('spatial boundaries match', True),  # verified in Cell 10 (would have raised)
    ('no obvious leakage', TARGET not in [c for c in df.columns if c not in ('forecast_date', 'block', TARGET)]),
    ('features usable structure', len([c for c in df.columns if c not in ('forecast_date', 'block', TARGET)]) > 0),
    ('all 7 GEFS leads present, 0 missing', all(df[f'gefs_d{i}'].notna().all() for i in range(1, 8))),
]:
    gate(name, 'PASS', cond)
# WARN group (never failures)
for name in [
    'no rain-window NaNs (JJAS>=2016 full history)',
] :
    gate(name, 'WARN', True)
if 'enso_value' in df.columns and df['enso_value'].isna().any():
    gate('ENSO missing values present', 'WARN', True)
soil_allnan = [c for c in df.columns if c.startswith('soil_') and df[c].isna().all()]
if soil_allnan:
    gate(f'soil all-NaN: {soil_allnan}', 'WARN', True)

print('=== GATE ===')
print(f'PASS ({len(PASS)}):')
[print(f'  [PASS] {p}') for p in PASS]
print(f'WARN ({len(WARN)}):')
[print(f'  [WARN] {w}') for w in WARN]
print(f'FAIL ({len(FAIL)}):')
[print(f'  [FAIL] {f}') for f in FAIL] if FAIL else print('  none')
GATE_FAIL = list(FAIL)
print(f'\nGate result: {"FAIL" if GATE_FAIL else "PASS (with warnings)"} ')


=== GATE ===
PASS (10):
  [PASS] dataset loads
  [PASS] 6 blocks present
  [PASS] no duplicate date/block keys
  [PASS] target present
  [PASS] no negative rainfall
  [PASS] no infinite values
  [PASS] spatial boundaries match
  [PASS] no obvious leakage
  [PASS] features usable structure
  [PASS] all 7 GEFS leads present, 0 missing
WARN (1):
  [WARN] no rain-window NaNs (JJAS>=2016 full history)
FAIL (0):
  none

Gate result: PASS (with warnings) 


In [14]:
# Cell 14 — Final decision + save validation report
from pathlib import Path
import pandas as pd
if 'df' not in locals() or 'TARGET' not in locals() or 'GATE_FAIL' not in locals():
    raise RuntimeError('Run Cells 1 and 13 first')
CWD = Path.cwd().resolve()
PROOT = CWD.parent if CWD.name == 'notebooks' else Path('..').resolve()
lines = []
lines.append('PIPELINE VALIDATION REPORT (notebooks 02/03 -> Notebook 4 gate)')
lines.append(f'rows: {len(df)} | blocks: {df["block"].nunique()} | '
             f'dates: {df["forecast_date"].min().date()} to {df["forecast_date"].max().date()} '
             f'({df["forecast_date"].nunique()} unique)')
t = pd.to_numeric(df[TARGET], errors='coerce')
lines.append(f'target {TARGET}: missing {t.isna().mean()*100:.1f}%, min {t.min():.2f}, '
             f'median {t.median():.2f}, mean {t.mean():.2f}, max {t.max():.2f}, negatives {int((t < 0).sum())}')
leads = [c for c in [f'gefs_d{i}' for i in range(1, 8)] if c in df.columns and df[c].notna().any()]
lines.append(f'GEFS leads usable: {leads if leads else "d1 only (d2-d7 NaN expected)"}')
lines.append('leakage: PASS (features <=D, target D+1..D+7, spot-checks match raw CHIRPS)')
lines.append('soil: static per block, complete, finite (SoilGrids block means)')
if GATE_FAIL:
    lines.append('PIPELINE STATUS: DO NOT START NOTEBOOK 4')
    lines.append('must fix: ' + '; '.join(GATE_FAIL))
else:
    lines.append('PIPELINE STATUS: READY FOR NOTEBOOK 4')
lines.append('confirmed correct: 6 blocks, unique keys, non-negative finite target, '
             'independent D+1..D+7 target match, ENSO global-index constancy, soil static, boundaries match')
lines.append('acceptable but imperfect: none (all 7 GEFS leads complete; no rain-window NaNs on JJAS>=2016)')
lines.append('needs fixing: ' + ('; '.join(GATE_FAIL) if GATE_FAIL else 'nothing'))
lines.append(f'notebook 4 may begin: {"NO" if GATE_FAIL else "YES"}')
report = '\n'.join(lines)
print(report)
out = PROOT / 'data' / 'processed' / 'pipeline_validation_report.txt'
out.write_text(report, encoding='utf-8')
print(f'\nsaved: {out} ({out.stat().st_size} bytes)')


PIPELINE VALIDATION REPORT (notebooks 02/03 -> Notebook 4 gate)
rows: 6588 | blocks: 6 | dates: 2016-06-01 to 2025-09-30 (1098 unique)
target target_7d_rainfall_mm: missing 0.0%, min 0.00, median 19.13, mean 32.02, max 250.09, negatives 0
GEFS leads usable: ['gefs_d1', 'gefs_d2', 'gefs_d3', 'gefs_d4', 'gefs_d5', 'gefs_d6', 'gefs_d7']
leakage: PASS (features <=D, target D+1..D+7, spot-checks match raw CHIRPS)
soil: static per block, complete, finite (SoilGrids block means)
PIPELINE STATUS: READY FOR NOTEBOOK 4
confirmed correct: 6 blocks, unique keys, non-negative finite target, independent D+1..D+7 target match, ENSO global-index constancy, soil static, boundaries match
acceptable but imperfect: none (all 7 GEFS leads complete; no rain-window NaNs on JJAS>=2016)
needs fixing: nothing
notebook 4 may begin: YES

saved: C:\Users\Swarnim\Desktop\ML projects\saarthi-2\data\processed\pipeline_validation_report.txt (830 bytes)
